# ModelV3 inference on a real MACSIMA image

Apply the v3 inversion model -- trained **only on synthetic data** -- to a real myocarditis MACSIMA
image. We predict the cell + nucleus instance segmentation (both from Omnipose flows) and the
generative parameters, then **render synthetic samples from those learned parameters**.

Uses `runs/run12/model_backsub_final.pt` -- the newest model (step 276000): per-marker normalisation + expression-strength, nucleus flows, background-only markers, the full cell/structure DENSITY spectrum (which it also PREDICTS), and trained in the **MACSIMA background-subtracted domain** (empty regions are near-zero speckle, not a pedestal) so it matches the staged `*_backsub.ome.tif` inputs. Runs in the `synmac` env (torch + scipy + tifffile + zarr + skimage). The
model is out-of-distribution on real data, so this is exploratory.

Runs on the **staged, background-subtracted** MACSIMA output (the real pipeline domain); the generator simulates that background subtraction so empty regions match and are not oversegmented.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))
import re, json
import numpy as np, torch, tifffile, zarr
from scipy import ndimage as ndi
from skimage.segmentation import find_boundaries
import matplotlib.pyplot as plt

import config as cfg, flexible_types as ft
from model import normalize_markers
from model_v3 import ModelV3
import gen_dataset as gd
# the SAME decoder the rounds evaluation scores with (short integration + sink clustering)
from flow_decode import flow_to_instances, match, relabel
from model_v3 import decode_queries
from parameter import (CellGeometry, CellType, MarkerPanel, P, dapi_marker,
                       narrow, from_u, Detector, TissueGeometry)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# the open-ended run13 (run12 + query instance head + wide prior) once it has a best checkpoint;
# until then run12. Re-run the notebook any time during training -- model_best.pt is refreshed on
# every round that improves the eval composite (runs/run13/rounds_log.jsonl has the history).
CKPT   = ("../runs/run13/model_best.pt" if os.path.exists("../runs/run13/model_best.pt")
          else "../runs/run12/model_backsub_final.pt")
# staged MACSIMA output (background-subtracted + clipped at 0) -- the domain the model is
# now trained on. Channel names come from MCMICRO backsub's markers_bs.csv beside the image (section 1);
# the DAPI rows there (Filter DAPI) are NOT background-subtracted.
IMAGE  = ("/mnt/Transfer/Myocarditis26/20260130_AGBorst_Myocarditis_Slide2_260216_420907_R1_A1_ROI2_staged/1/rack-01-well-A01-roi-002-exp-2/background/rack-01-well-A01-roi-002-exp-2_backsub.ome.tif")

ck = torch.load(CKPT, map_location=DEVICE, weights_only=False)
model = ModelV3(n_global=ck["cfg"]["n_global"], n_marker_noise=4, K=ck["cfg"]["k_slots"], T=3,
                n_queries=ck["cfg"].get("n_queries", 0),
                use_flow=ck["cfg"].get("use_flow", True)).to(DEVICE).eval()
model.load_state_dict(ck["model"])
HAS_Q = model.n_queries > 0          # query-based instance head present in this checkpoint?
# run13 from round 21 trains the query head ONLY (--no-flow): its flow convs are untrained, so the
# flow decodes below are skipped and the query instances stand in for them everywhere
HAS_FLOW = model.use_flow
print(f"loaded checkpoint at step {ck['step']}  |  device {DEVICE}  |  instance head: "
      f"{'%d queries' % model.n_queries if HAS_Q else 'none (flows only)'}  |  flows: "
      f"{'trained' if HAS_FLOW else 'off (query-only)'}")

## 1. Real crops at the training CELL scale

The model trained on 160 x 160 px tiles. What has to match is how many pixels a cell spans, not the
micron label: measured with the same estimator (Otsu on the nuclear stain), training nuclei are
~2.3 um (~7 px) across while the real nuclei are ~5.4 um, so a crop cut at the nominal 0.325 um/px
shows the model cells ~2.3x bigger than anything it has seen (and only ~5x5 of them).

`SCALE_MODE = "cells"` (default) therefore sizes each crop so the real nuclear diameter in pixels
matches the training one: the raw window `S` is zoomed to 160 px, with `S` measured from the crops
themselves against the run's eval bundles. `"microns"` restores the physical geometry (0.17 um raw ->
0.325 um, 306 raw px). Several crops are taken so the real side is not one sample: the first is the
original region, the rest are tissue positions picked at random.

In [ ]:
T = 160                                                 # training tile, px
_tf = tifffile.TiffFile(IMAGE)
UM_RAW = float(re.search(r'PhysicalSizeX="([0-9.]+)"', _tf.ome_metadata).group(1))
# Channel names come from MCMICRO backsub's markers_bs.csv, written beside the image: its rows ARE
# the output channels, in order (background channels already removed). The OME names are only a
# cross-check -- the csv is what backsub actually wrote.
import csv
MARKERS_CSV = os.path.join(os.path.dirname(IMAGE), "markers_bs.csv")
with open(MARKERS_CSV) as _f:
    MARKER_ROWS = list(csv.DictReader(_f))
CH_NAMES = [r["marker_name"] for r in MARKER_ROWS]
_ome_names = re.findall(r'<Channel ID="[^"]*"[^>]*?Name="([^"]*)"', _tf.ome_metadata)
if _ome_names and _ome_names != CH_NAMES:
    print(f"WARNING: OME channel names differ from {MARKERS_CSV} -- using the csv")

# top-left corners at full resolution
REAL_CROPS = [(10598, 28365),   # the original tissue-rich region (centre of the old 1024 px crop)
              (6471, 17831), (3303, 18743), (2439, 24103), (17687, 7559)]
z = zarr.open(tifffile.imread(IMAGE, aszarr=True), mode="r")
full = z["0"] if hasattr(z, "keys") else z
M = full.shape[0]
assert M == len(CH_NAMES), (f"image has {M} channels but {MARKERS_CSV} lists {len(CH_NAMES)} -- "
                            f"point MARKERS_CSV at the markers_bs.csv of THIS backsub run")

from skimage.filters import threshold_otsu
def _otsu_fg(d):
    g = ndi.gaussian_filter(d.astype(np.float32), 1.0)
    return g > threshold_otsu(g)

def _nuc_contrast(d):
    fg = _otsu_fg(d)
    return float(np.median(d[~fg]) / (np.median(d[fg]) + 1e-6))

def _nuc_diam(d, min_px=10):
    """median equivalent diameter (px) of the Otsu nuclei on a nuclear-stain image -- the SAME crude,
    model-free estimator on both sides, so its bias (it under-calls GT nuclei ~0.7x) cancels."""
    lab, _ = ndi.label(_otsu_fg(d))
    a = np.bincount(lab.ravel())[1:]
    a = a[a >= min_px]
    return 2.0 * np.sqrt(a / np.pi) if a.size else np.zeros(0)

# Probe windows at native resolution -- nuclear-stain channels only (all 48 at this size would be
# ~1.5 GB) -- wide enough to hold the crop at any plausible scale.
PROBE = 1400
DAPI_CANDS = [i for i, r in enumerate(MARKER_ROWS) if r["Filter"].strip().upper() == "DAPI"] or [0]
probes = [{i: np.asarray(full[i, r0:r0 + PROBE, c0:c0 + PROBE]).astype(np.float32) for i in DAPI_CANDS}
          for r0, c0 in REAL_CROPS]

# The nuclear channel. A slide can carry SEVERAL DAPI acquisitions (Filter DAPI in the csv): on the
# Slide1-3 runs the cycle-1 "DAPI" is mostly blue tissue autofluorescence (off/on-nucleus ~0.6-0.7,
# nuclei ~1-2k ADU) and the re-stain "DAPI_cycle5" is the real nuclear stain (~0.03-0.06, nuclei
# 30-45k ADU). So pick, among the DAPI rows, the one with the sharpest nuclei on these crops. Every
# channel -- including the weak DAPI -- still goes into the model; this only decides what is shown
# and scored AS the nuclear stain.
_dcon = {i: float(np.median([_nuc_contrast(p[i][:600, :600]) for p in probes])) for i in DAPI_CANDS}
DAPI_CH = min(_dcon, key=_dcon.get)
NON_DAPI = [i for i in range(M) if i not in DAPI_CANDS]   # antibody channels only
print("DAPI candidates (off/on-nucleus, lower = sharper): " +
      ", ".join(f"{CH_NAMES[i]} (ch {i}) {c:.2f}" for i, c in _dcon.items()) +
      f"  ->  nuclear channel = {CH_NAMES[DAPI_CH]} (ch {DAPI_CH})")

# How big the model sees a cell. "cells": zoom so a real nucleus spans as many px as a training one
# (training reference = Otsu on the counterstain -- canonical marker 0 -- of the run's eval bundles;
# 7.1 px is what run13's ORIGINAL small-cell eval set measured, used if no bundles are on disk;
# since 2026-11-10 the eval set spans the widened cell-size prior, so this tracks its median). "microns": the
# nominal physical geometry. They differ because the generator's cells are physically SMALLER than
# these real ones (training nuclei ~2.3 um by this estimator vs ~5.4 um real at the same estimator).
SCALE_MODE = "cells"
TRAIN_REF = "../data/rounds_run13/eval"
def _train_nuc_diam_px(ref_dir, n=120, default=7.1):
    import glob
    ds_ = []
    for f in sorted(glob.glob(os.path.join(ref_dir, "*.npz")))[:n]:
        b = np.load(f)
        di = np.flatnonzero(b["th_channel_order"] == 0)
        ds_.append(_nuc_diam(b["image"][..., int(di[0]) if di.size else 0]))
    ds_ = np.concatenate(ds_) if ds_ else np.zeros(0)
    return (float(np.median(ds_)), ds_.size) if ds_.size else (default, 0)

TRAIN_NUC_PX, _n_ref = _train_nuc_diam_px(TRAIN_REF)
REAL_NUC_RAW = float(np.median(np.concatenate([_nuc_diam(p[DAPI_CH]) for p in probes])))
S_MICRONS = int(round(T * cfg.UM_PER_VOX / UM_RAW))     # raw px covering 52 um (306)
# First guess from the native-resolution ratio, then a few fixed-point steps measured AFTER the zoom:
# downsampling blurs neighbouring nuclei together, so the Otsu diameter does not scale exactly with 1/S.
_zoomed = lambda S_, i=None: [ndi.zoom(p[DAPI_CH if i is None else i][:S_, :S_], T / S_, order=1)[:T, :T]
                              for p in probes]
_diam_at = lambda S_: float(np.median(np.concatenate([_nuc_diam(d) for d in _zoomed(S_)])))
S_CELLS = int(round(T * REAL_NUC_RAW / TRAIN_NUC_PX))    # raw px holding as many nuclei as a training tile
for _ in range(4):
    S_CELLS = int(np.clip(round(S_CELLS * _diam_at(S_CELLS) / TRAIN_NUC_PX), T, PROBE))
S = S_CELLS if SCALE_MODE == "cells" else S_MICRONS
assert S <= PROBE, f"crop S={S} exceeds the probe window -- raise PROBE to at least {S}"
print(f"nucleus diameter: real {REAL_NUC_RAW:.1f} raw px ({REAL_NUC_RAW * UM_RAW:.1f} um) | training "
      f"{TRAIN_NUC_PX:.1f} px ({TRAIN_NUC_PX * cfg.UM_PER_VOX:.1f} um; {_n_ref} nuclei from {TRAIN_REF})")
print(f"SCALE_MODE={SCALE_MODE!r}: S = {S} raw px -> {T} px  (cells: {S_CELLS}, microns: {S_MICRONS}); "
      f"effective {UM_RAW * S / T:.3f} um/px vs nominal {cfg.UM_PER_VOX}")

reals = [ndi.zoom(np.asarray(full[:, r0:r0 + S, c0:c0 + S]).astype(np.float32), (1, T / S, T / S),
                  order=1)[:, :T, :T] for r0, c0 in REAL_CROPS]              # [M, 160, 160] backsub ADU
ds = reals[0]                                            # sections 5+ use the first crop
print(f"{len(reals)} real crops: {S} raw px at {UM_RAW} um -> {T} px  |  {M} markers  |  "
      f"nucleus diameter after zoom {np.median(np.concatenate([_nuc_diam(r[DAPI_CH]) for r in reals])):.1f} px "
      f"(training {TRAIN_NUC_PX:.1f})")

## 2. Per-marker normalisation + inference

Each marker is normalised to [0,1] and a per-marker **expression strength** is computed (the same
`normalize_markers` the model trains with), so a 10-100x brighter DAPI no longer dominates and the
model knows each channel's relative brightness. Every crop is run; the parameter read-out and the
samples in section 5 use the first crop.

In [ ]:
def run_model(img_hwm):
    """Same preprocessing as the training collate: per-marker normalise + strength. [H, W, M] in."""
    norm, strn = normalize_markers(img_hwm)
    with torch.no_grad():
        o = model(torch.from_numpy(norm).permute(2, 0, 1)[None].to(DEVICE),
                  present=torch.ones(1, norm.shape[-1], dtype=torch.bool, device=DEVICE),
                  strength=torch.from_numpy(strn)[None].to(DEVICE))
    d = o["dense"]
    return dict(cell=torch.sigmoid(d["cell"][0, 0]).cpu().numpy(),
                nuc=torch.sigmoid(d["nucleus"][0, 0]).cpu().numpy(),
                flow=d["flow"][0].cpu().numpy() if "flow" in d else None,
                nflow=d["nuc_flow"][0].cpu().numpy() if "nuc_flow" in d else None,
                strength=strn, norm=norm, out=o, inst=o.get("inst"))

real_pred = [run_model(np.moveaxis(r, 0, -1)) for r in reals]

p0 = real_pred[0]
out, norm, strength = p0["out"], p0["norm"], p0["strength"]
nuc, cell, flow, nuc_flow = p0["nuc"], p0["cell"], p0["flow"], p0["nflow"]
so = out["slots"][-1]
exist = so["existence"][0].sigmoid().cpu().numpy()
present = [k for k in range(len(exist)) if exist[k] > 0.5]
for i, p in enumerate(real_pred):
    n_types = int((p["out"]["slots"][-1]["existence"][0].sigmoid() > 0.5).sum())
    print(f"crop {i}: {n_types} cell types  |  strength [{p['strength'].min():.2f}, "
          f"{p['strength'].max():.2f}]  |  mean cell prob {p['cell'].mean():.2f}")

# newest model also predicts the global tissue DENSITY (prepended at global indices 0-3)
g_u = out["global_mu"][0].sigmoid().cpu().numpy()         # normalised [0,1]
_tg = TissueGeometry()
pred_min_dist = from_u(_tg.MIN_DIST, float(g_u[0]))      # cell spacing, voxels (small = dense)
pred_cover    = from_u(_tg.COVER,    float(g_u[1]))      # tissue coverage (1 = no background)
pred_grow     = from_u(_tg.GROW,     float(g_u[2]))      # confluence
pred_struct   = float(g_u[3])                            # structure territory fraction
print(f"crop 0 predicted density: MIN_DIST~{pred_min_dist:.1f} vox  COVER~{pred_cover:.2f}  "
      f"GROW~{pred_grow:.2f}  structure_frac~{pred_struct:.2f}")

## 3. The real crops

In [ ]:
def nrm(a, p=99.5):
    a = a.astype(np.float32); hi = np.percentile(a, p)
    return np.clip(a / (hi + 1e-6), 0, 1)

def _ch(name, k):
    """channel index of a marker by its markers_bs.csv name; the k-th antibody channel if absent"""
    return CH_NAMES.index(name) if name in CH_NAMES else NON_DAPI[k % len(NON_DAPI)]
# nuclear stain + three markers with a known look in myocardium: cardiomyocytes, leukocytes, matrix
SHOW = [DAPI_CH, _ch("Cardiac-Troponin-T", 0), _ch("CD45", 1), _ch("CollagenI", 2)]
RGB = (SHOW[1], SHOW[2], DAPI_CH)                       # R, G, B(=nuclear stain)
real_rgbs = [np.stack([nrm(r[c]) for c in RGB], -1) for r in reals]
real_rgb = real_rgbs[0]

fig, ax = plt.subplots(len(reals), 5, figsize=(18, 3.7 * len(reals)), squeeze=False)
for r, crop in enumerate(reals):
    for i, c in enumerate(SHOW):
        ax[r, i].imshow(nrm(crop[c]), cmap="gray")
        ax[r, i].set_title(f"crop {r}: {CH_NAMES[c]} (ch {c})")
    ax[r, 4].imshow(real_rgbs[r]); ax[r, 4].set_title("RGB " + " / ".join(CH_NAMES[c] for c in RGB))
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.show()

## 4. Predicted segmentation on the real crops (nuclei + cells, both from flows)

Decoded with `flow_decode.flow_to_instances` -- the same decoder the rounds evaluation scores with.

With a query-head checkpoint (`HAS_Q`) a fifth column shows the per-cell instances of that head --
no flows, one query per cell, each with its own nucleus, type slot and context.

In [ ]:
ncol = (6 if HAS_Q else 5) if HAS_FLOW else 4      # +1: the plain real RGB crop on the left
fig, ax = plt.subplots(len(reals), ncol, figsize=(4 * ncol, 4.1 * len(reals)), squeeze=False)
for r, p in enumerate(real_pred):
    ax[r, 0].imshow(real_rgbs[r])
    ax[r, 0].set_title(f"crop {r}: " + " / ".join(CH_NAMES[c] for c in RGB), fontsize=9)
    ax[r, 1].imshow(p["nuc"], cmap="magma", vmin=0, vmax=1); ax[r, 1].set_title(f"crop {r}: nucleus prob")
    if HAS_Q:
        p["q_cell"], p["q_nuc"], p["q_props"] = decode_queries(p["inst"], 0)
        qv = real_rgbs[r].copy()
        qv[find_boundaries(p["q_cell"], mode="inner")] = [0, 1, 1]
        qv[find_boundaries(p["q_nuc"], mode="inner")] = [1, 0, 1]
    if HAS_FLOW:
        p["nuc_inst"] = flow_to_instances(p["nflow"], p["nuc"] > 0.5, device=DEVICE)
        p["cell_inst"] = flow_to_instances(p["flow"], p["cell"] > 0.5, device=DEVICE)
        ov = real_rgbs[r].copy(); ov[find_boundaries(p["cell_inst"], mode="inner")] = [1, 1, 0]
        ax[r, 2].imshow(p["nuc_inst"], cmap="nipy_spectral"); ax[r, 2].set_title(f"nuclei (flows): {p['nuc_inst'].max()}")
        ax[r, 3].imshow(p["cell_inst"], cmap="nipy_spectral"); ax[r, 3].set_title(f"cells (flows): {p['cell_inst'].max()}")
        ax[r, 4].imshow(ov); ax[r, 4].set_title("flow cells on image")
        if HAS_Q:
            ax[r, 5].imshow(qv); ax[r, 5].set_title(f"query head: {p['q_cell'].max()} cells (cyan), nuclei (magenta)")
    else:
        # query-only checkpoint: the query instances ARE the segmentation
        p["nuc_inst"], p["cell_inst"] = p["q_nuc"], p["q_cell"]
        ax[r, 2].imshow(p["q_cell"], cmap="nipy_spectral"); ax[r, 2].set_title(f"cells (query): {p['q_cell'].max()}")
        ax[r, 3].imshow(qv); ax[r, 3].set_title(f"query head: {p['q_cell'].max()} cells (cyan), nuclei (magenta)")
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.show()
nuc_inst, cell_inst = real_pred[0]["nuc_inst"], real_pred[0]["cell_inst"]

In [ ]:
# query-head instances as QuPath cell objects (body + nucleus + type slot + context), mapped back to
# full-resolution slide coordinates so they overlay the original image
from geojson_export import build_prediction_geojson
if HAS_Q:
    os.makedirs("../renders", exist_ok=True)
    for r, ((r0, c0), p) in enumerate(zip(REAL_CROPS, real_pred)):
        gjs = build_prediction_geojson(p["q_cell"], p["q_nuc"], p["q_props"],
                                       context_names=["background"] + list(cfg.CONTEXT_KINDS),
                                       scale=S / T, offset_xy=(c0, r0))
        path = f"../renders/pred_crop{r}_r{r0}_c{c0}.geojson"
        json.dump(gjs, open(path, "w"))
        print(f"crop {r}: {len(gjs['features'])} cells -> {path}")
else:
    print("checkpoint has no instance head -- nothing to export (flow instances carry no type)")

## 4b. Control: the same model on held-out synthetic tiles (with ground truth)

The real image has no ground truth, so a bad segmentation there cannot tell you *why*. This block
runs the identical preprocessing + model + `flow_to_instances` on fresh synthetic tiles drawn exactly
like the run12 training bundles (flexible world, final curriculum, no artifacts, unseen seeds) and
scores them against the generator's own masks.

Reading the outcome:
- **synthetic bad** -> the model (or the decoder) does not do the job even in-distribution; real data is not the issue yet.
- **synthetic good, real bad** -> domain gap; section 4c shows which input statistics differ.

The four decodes per tile split the blame: `GT fg + GT flow` is the ceiling of `flow_to_instances`
itself (a post-processing problem if this is low), `GT fg + pred flow` isolates the flow head,
`pred fg + GT flow` isolates the foreground head, `pred fg + pred flow` is what section 4 does.

In [ ]:
import flexible_types as ft
from derived_targets import instance_flows

# Fresh synthetic tiles drawn EXACTLY as the training bundles are (run_v3 gen -> build_bundle ->
# draw_flexible): flexible world at the final curriculum, no artifacts, same seeding scheme -- but a
# seed far outside anything run12 trained on, so these are held-out samples from the training prior.
N_SYN, SYN_SEED = 6, 987_000_000
# True = render exactly the distribution run12 was TRAINED on (build_flexible_world prior="run12":
# no tissue background, DAPI on the antibody range, 100 ADU DAPI pedestal, old density / shape /
# panel ranges, default textures, one detector) -- the honest in-distribution control for a run12
# checkpoint. False = the current wide prior (cfg.WIDE_PRIOR), i.e. what run13 trains on.
LIKE_RUN12 = "run12" in CKPT
syn = []
for i in range(N_SYN):
    for attempt in range(5):                       # draw_flexible's retry on an infeasible architecture
        off = attempt * 1_000_000_000
        try:
            w = ft.build_flexible_world(np.random.default_rng(SYN_SEED + 500_000 + i + off),
                                        curriculum=cfg.curriculum_at(1.0), dyes=cfg.DYES,
                                        detector=Detector(**cfg.DETECTOR),
                                        prior="run12" if LIKE_RUN12 else "wide")
            img, _, _, _, lab = gd.render_frame(w, seed=SYN_SEED + 10_000_000 + i + off,
                                                pool_bank=cfg.POOL_BANK, with_artifacts=False)
            break
        except ValueError as e:
            print(f"  synth {i}: retry ({e})")
    vis = [int(r["id"]) for r in lab["cells"] if bool(r["visible"])]
    inst = np.where(np.isin(lab["inst"], vis), lab["inst"], 0)
    # the nucleus target is EVERY nucleus in the frame (bundle "nuc"), not just visible cells' --
    # labelled by its cell id, as in training
    nuc_gt = np.where(lab["nuc"] > 0, lab["inst"], 0)
    syn.append(dict(img=img, names=list(w["PANEL"].Markers), cell_gt=relabel(inst),
                    nuc_gt=relabel(nuc_gt),
                    flow_gt=instance_flows(inst, vis), nflow_gt=instance_flows(nuc_gt, np.unique(nuc_gt[nuc_gt > 0]))))
    print(f"synth {i}: {img.shape[0]}x{img.shape[1]} px, {img.shape[2]} markers, "
          f"{len(vis)} visible cells, DAPI = channel {syn[-1]['names'].index('DAPI')}")

In [ ]:
def sem_counts(a, b):
    return int((a & b).sum()), int((a | b).sum())


# Four decodes per tile separate the failure modes:
#   GT fg + GT flow     -> ceiling of flow_to_instances itself (post-processing, not the network)
#   GT fg + pred flow   -> is the predicted FLOW good, given a perfect foreground?
#   pred fg + GT flow   -> is the predicted FOREGROUND good, given perfect flows?
#   pred fg + pred flow -> what the notebook actually does on the real image
rows = []
for s in syn:
    p = run_model(s["img"]); s["pred"] = p
    for obj, gt, fgt, pfg, pfl in (("cell", s["cell_gt"], s["flow_gt"], p["cell"], p["flow"]),
                                   ("nucleus", s["nuc_gt"], s["nflow_gt"], p["nuc"], p["nflow"])):
        gfg, pf = gt > 0, pfg > 0.5
        dec = {"GT fg + GT flow": (gfg, fgt), "GT fg + pred flow": (gfg, pfl),
               "pred fg + GT flow": (pf, fgt), "pred fg + pred flow": (pf, pfl)}
        for name, (fg, fl) in (dec.items() if HAS_FLOW else ()):   # flows untrained -> no flow rows
            inst = flow_to_instances(fl, fg, device=DEVICE)
            m = match(gt, inst)
            inter, union = sem_counts(gfg, pf)
            rows.append(dict(obj=obj, decode=name, inter=inter, union=union, tp=m["rec"] * m["n_gt"],
                             miou_sum=m["miou"] * m["rec"] * m["n_gt"], **m))
            if name == "pred fg + pred flow":
                s[obj[:3] + "_pred_inst"] = inst
    if HAS_Q:                                       # the query head needs no flows at all
        qc, qn, _ = decode_queries(p["inst"], 0)
        for obj, gt, q_ in (("cell", s["cell_gt"], qc), ("nucleus", s["nuc_gt"], qn)):
            m = match(gt, q_)
            inter, union = sem_counts(gt > 0, q_ > 0)
            rows.append(dict(obj=obj, decode="query head", inter=inter, union=union,
                             tp=m["rec"] * m["n_gt"], miou_sum=m["miou"] * m["rec"] * m["n_gt"], **m))
            if not HAS_FLOW:                        # the query instances feed the plots below
                s[obj[:3] + "_pred_inst"] = q_

# POOLED over tiles (sum the counts, then divide): a per-tile mean lets one empty tile -- 0 cells,
# F1 0 by convention -- drag every row down
print(f"{'object':8s} {'decode':22s} {'fg IoU':>7s} {'n_gt':>5s} {'n_pred':>6s} "
      f"{'prec':>5s} {'rec':>5s} {'F1@.5':>6s} {'mIoU':>5s}   (pooled over {len(syn)} synthetic tiles)")
for obj in ("cell", "nucleus"):
    for name in (("GT fg + GT flow", "GT fg + pred flow", "pred fg + GT flow", "pred fg + pred flow")
                 if HAS_FLOW else ()) + (("query head",) if HAS_Q else ()):
        r = [x for x in rows if x["obj"] == obj and x["decode"] == name]
        S_ = {k: sum(x[k] for x in r) for k in ("inter", "union", "n_gt", "n_pred", "tp", "miou_sum")}
        print(f"{obj:8s} {name:22s} {S_['inter'] / max(S_['union'], 1):7.2f} {S_['n_gt']:5d} "
              f"{S_['n_pred']:6d} {S_['tp'] / max(S_['n_pred'], 1):5.2f} {S_['tp'] / max(S_['n_gt'], 1):5.2f} "
              f"{2 * S_['tp'] / max(S_['n_gt'] + S_['n_pred'], 1):6.2f} {S_['miou_sum'] / max(S_['tp'], 1e-12):5.2f}")

In [ ]:
fig, ax = plt.subplots(len(syn), 5, figsize=(19, 3.9 * len(syn)), squeeze=False)
for r, s in enumerate(syn):
    di = s["names"].index("DAPI")
    mk = [c for c in np.argsort(s["img"].reshape(-1, s["img"].shape[2]).mean(0))[::-1] if c != di]
    rgb = np.stack([nrm(s["img"][..., mk[0]]), nrm(s["img"][..., mk[1]]), nrm(s["img"][..., di])], -1)
    gt_ov, pr_ov = rgb.copy(), rgb.copy()
    gt_ov[find_boundaries(s["cell_gt"], mode="inner")] = [1, 1, 0]
    pr_ov[find_boundaries(s["cel_pred_inst"], mode="inner")] = [0, 1, 1]
    ax[r, 0].imshow(rgb); ax[r, 0].set_title(f"synthetic #{r} (2 brightest + DAPI)")
    ax[r, 1].imshow(gt_ov); ax[r, 1].set_title(f"GT cells: {s['cell_gt'].max()}")
    ax[r, 2].imshow(pr_ov); ax[r, 2].set_title(f"pred cells: {len(np.unique(s['cel_pred_inst'])) - 1}")
    ax[r, 3].imshow(s["pred"]["cell"], cmap="magma", vmin=0, vmax=1); ax[r, 3].set_title("pred cell prob")
    ax[r, 3].contour(s["cell_gt"] > 0, [0.5], colors="w", linewidths=0.6)
    ax[r, 4].imshow(s["nuc_pred_inst"], cmap="nipy_spectral")
    ax[r, 4].contour(s["nuc_gt"] > 0, [0.5], colors="w", linewidths=0.6)
    ax[r, 4].set_title(f"pred nuclei {len(np.unique(s['nuc_pred_inst'])) - 1} (white = GT {s['nuc_gt'].max()})")
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.show()

## 4c. Real vs synthetic: where do the inputs differ?

Input statistics in the space the model sees (after `normalize_markers`), all real crops pooled: DAPI
and marker intensity histograms, the `strength` input, the confidence of the cell head, and nucleus
size. Where the real image has no truth, the same model-free estimator (Otsu on DAPI) runs on both
sides. The table underneath is per channel -- it is where an unexpressed synthetic marker (pure
clipped read noise, ~50 ADU levels) shows up against a real one.

`LIKE_RUN12` (4b generation cell) follows the checkpoint: a run12 checkpoint is tested on the run12
prior, a run13 one on the wide prior it trains on. Override it to compare across.

In [ ]:
# input statistics real vs synthetic, in the space the model actually sees. The SAME estimator
# runs on both sides wherever there is no ground truth on the real image (Otsu on smoothed DAPI).
# Real side: the nuclear channel is DAPI_CH (section 1), and "markers" excludes every DAPI row.
from skimage.filters import threshold_otsu

def otsu_dapi(d):
    """Crude, model-free nuclei on a DAPI channel: Otsu on a sigma-1 blur. Returns (labels, fg)."""
    g = ndi.gaussian_filter(d.astype(np.float32), 1.0)
    fg = g > threshold_otsu(g)
    return ndi.label(fg)[0], fg

def areas(lab, min_px=10):
    a = np.bincount(lab.ravel())[1:]
    return a[a >= min_px]                          # drop single-pixel speckle blobs

def dapi_contrast(d, fg, tissue=None):
    """median DAPI off-nucleus / median on-nucleus: ~0 = dark background, -> 1 = no contrast.
    `tissue` restricts "off" to tissue -- the real crops are all tissue, a synthetic tile often is
    not, and glass would drag its ratio toward 0."""
    off = ~fg if tissue is None else (~fg & tissue)
    return float(np.median(d[off]) / (np.median(d[fg]) + 1e-6)) if off.any() and fg.any() else np.nan

real_norm = np.stack([p["norm"] for p in real_pred])          # [n_crops, H, W, M]
real_str = np.concatenate([p["strength"] for p in real_pred])
st = dict(syn_dapi=[], syn_mk=[], syn_str=[], syn_cellp=[], a_gt=[], a_otsu=[], a_pred=[], con=[], z_in=[])
for s in syn:
    di, nm = s["names"].index("DAPI"), s["pred"]["norm"]
    tis = s["cell_gt"] > 0
    st["syn_dapi"].append(nm[..., di].ravel())
    st["syn_mk"].append(np.delete(nm, di, -1).ravel())
    st["syn_str"].append(s["pred"]["strength"])
    st["syn_cellp"].append(s["pred"]["cell"].ravel())
    st["a_gt"].append(areas(s["nuc_gt"]))
    st["a_pred"].append(areas(s["nuc_pred_inst"]))
    lab, fg = otsu_dapi(s["img"][..., di]); st["a_otsu"].append(areas(lab))
    st["con"].append(dapi_contrast(s["img"][..., di].astype(np.float32), fg, tissue=tis))
    if tis.any():                                  # zeros INSIDE tissue only (real crop is all tissue)
        st["z_in"].append(np.mean(np.delete(s["img"], di, -1)[tis] == 0))
cat = {k: np.concatenate(v) for k, v in st.items() if k not in ("con", "z_in")}
_otsu = [otsu_dapi(r[DAPI_CH]) for r in reals]
real_a_otsu = np.concatenate([areas(lab) for lab, _ in _otsu])
real_a_pred = np.concatenate([areas(p["nuc_inst"]) for p in real_pred])
real_con = [dapi_contrast(r[DAPI_CH], fg) for r, (_, fg) in zip(reals, _otsu)]

fig, ax = plt.subplots(1, 5, figsize=(22, 3.8))
kw = dict(bins=60, density=True, histtype="step", lw=1.5)
ax[0].hist(cat["syn_dapi"], label="synthetic", **kw); ax[0].hist(real_norm[..., DAPI_CH].ravel(), label="real", **kw)
ax[0].set_yscale("log"); ax[0].set_title("DAPI, normalised intensity"); ax[0].legend()
ax[1].hist(cat["syn_mk"], **kw); ax[1].hist(real_norm[..., NON_DAPI].ravel(), **kw)
ax[1].set_yscale("log"); ax[1].set_title("other markers, normalised intensity")
ax[2].hist(cat["syn_str"], label="synthetic", **kw); ax[2].hist(real_str, label="real", **kw)
ax[2].set_title("per-marker strength input")
ax[3].hist(cat["syn_cellp"], **kw); ax[3].hist(np.concatenate([p["cell"].ravel() for p in real_pred]), **kw)
ax[3].set_yscale("log"); ax[3].set_title("pred cell prob (confident = bimodal)")
b = np.linspace(1, 3.5, 40)
for v, lb, ls in ((cat["a_gt"], "synth GT", "-"), (cat["a_otsu"], "synth DAPI Otsu", ":"),
                  (cat["a_pred"], "synth pred", "--"), (real_a_otsu, "real DAPI Otsu", ":"),
                  (real_a_pred, "real pred", "--")):
    ax[4].hist(np.log10(v), bins=b, density=True, histtype="step", lw=1.5, ls=ls, label=lb)
ax[4].set_xlabel("log10 nucleus area (px, >= 10)"); ax[4].set_title("nucleus size"); ax[4].legend(fontsize=7)
plt.tight_layout(); plt.show()

print(f"marker zero fraction INSIDE tissue (backsub floor): synthetic {np.mean(st['z_in']):.2f} | "
      f"real {np.mean([np.mean(r[NON_DAPI] == 0) for r in reals]):.2f}")
print(f"DAPI contrast (off/on nucleus inside tissue, Otsu): synthetic {np.nanmean(st['con']):.2f} | "
      f"real {np.mean(real_con):.2f} (per crop {np.round(real_con, 2).tolist()})")
print("nucleus area median (px): " + " | ".join(
    f"{lb} {np.median(v):.0f}" for lb, v in (("synth GT", cat["a_gt"]), ("synth Otsu", cat["a_otsu"]),
    ("synth pred", cat["a_pred"]), ("real Otsu", real_a_otsu), ("real pred", real_a_pred))))
print(f"markers: synthetic {[s['img'].shape[2] for s in syn]} | real {M}")
_p = lambda img, di: np.percentile(np.delete(img, di, -1).reshape(-1, img.shape[-1] - 1), 99.5, axis=0)
print(f"marker p99.5 ADU, median over channels (the comb in the synthetic histogram = few ADU levels): "
      f"synthetic {np.median(np.concatenate([_p(s['img'], s['names'].index('DAPI')) for s in syn])):.0f} | "
      f"real {np.median([np.percentile(r[NON_DAPI].reshape(len(NON_DAPI), -1), 99.5, axis=1) for r in reals]):.0f}")

# per channel: what each marker channel looks like in ADU (real: all crops; synthetic: all tiles)
def chan_stats(img_hwm, tissue):
    x = img_hwm[tissue] if tissue is not None and tissue.any() else img_hwm.reshape(-1, img_hwm.shape[-1])
    return np.stack([np.median(x, 0), np.percentile(x, 99.5, 0), (x == 0).mean(0),
                     [len(np.unique(x[:, k])) for k in range(x.shape[1])]], 1)   # [M, 4]

syn_rows = np.concatenate([np.delete(chan_stats(s["img"].astype(np.float32), s["cell_gt"] > 0),
                                     s["names"].index("DAPI"), 0) for s in syn])
real_rows = np.concatenate([chan_stats(np.moveaxis(np.round(r), 0, -1), None)[NON_DAPI] for r in reals])
q = lambda a: " / ".join(f"{v:.0f}" if v >= 1 else f"{v:.2f}" for v in np.percentile(a, [10, 50, 90]))
print("\nmarker channels, 10th / 50th / 90th percentile over channels (synthetic: inside tissue)")
print(f"{'':12s} {'tissue median ADU':>22s} {'p99.5 ADU':>22s} {'zero fraction':>20s} {'distinct levels':>22s}")
for lb, rows in (("synthetic", syn_rows), ("real", real_rows)):
    print(f"{lb:12s} {q(rows[:, 0]):>22s} {q(rows[:, 1]):>22s} {q(rows[:, 2]):>20s} {q(rows[:, 3]):>22s}")

## 5. Samples generated from the learned parameters

Reconstruct a renderable *world* from the predicted parameters (per-marker `mu`/texture, each
inferred cell type's geometry + marker expression) and render synthetic tiles -- the tissue as the
model understood it. Subset of markers for speed; placement is re-sampled so it won't match the crop.

In [ ]:
mo = out["marker"]
mu_u    = mo["mu"][0].sigmoid().cpu().numpy()
noise_u = mo["noise_mu"][0].sigmoid().cpu().numpy()
pos  = so["expression"]["positive"][0].sigmoid().cpu().numpy()
lvl  = so["expression"]["level_mu"][0].sigmoid().cpu().numpy()
geom = so["geometry"]["mu"][0].sigmoid().cpu().numpy()

M = mu_u.shape[0]                                   # channels in THIS image (panel varies per image)
_strength = (pos[present] * lvl[present]).max(0) if len(present) else np.zeros(M)
SEL = [int(c) for c in np.argsort(_strength)[::-1][:6]]   # the 6 most-expressed channels (always in range)
b_min, b_max = ft.expression_range(); LGL, LGH = np.log10(b_min), np.log10(b_max)
GLO, GHI = ft._geometry_bounds()

def unnorm_geom(u):
    g = {}
    for i, k in enumerate(ft.GEOM_KEYS):
        g[k] = (float(np.exp(u[i]*(np.log(GHI['RADIUS'])-np.log(GLO['RADIUS']))+np.log(GLO['RADIUS'])))
                if k == 'RADIUS' else GLO[k] + u[i]*(GHI[k]-GLO[k]))
    return CellGeometry(**{k: float(v) for k, v in g.items()})

markers = {"DAPI": dapi_marker(name="DAPI")}
for c in SEL:
    mu_v = cfg.MU_LO + float(mu_u[c]) * (cfg.MU_HI - cfg.MU_LO)
    mk = ft.build_marker(np.random.default_rng(c), name=f"M{c}", dye=cfg.DYES[c % 3],
                         mu=mu_v, noise_kind_idx=[1])
    comp = mk.noise_components[0]
    for j, key in enumerate(["scale", "clust", "fill", "s"]):
        p = getattr(comp, key, None)
        if isinstance(p, P):
            setattr(comp, key, narrow(p, p.lo, p.hi, from_u(p, float(noise_u[c, j]))))
    markers[f"M{c}"] = mk
panel = MarkerPanel(Markers=markers)

types = {}
for i, k in enumerate(present):
    expr = {f"M{c}": P(0., b_max, .05, float(10**(LGL + float(lvl[k, c])*(LGH-LGL))), "e")
            for c in SEL if pos[k, c] > 0.5}
    # every nucleus takes DAPI, as in gen_dataset.build_cell_types -- without it the DAPI channel
    # renders as pure detector noise (no nuclei)
    expr["DAPI"] = P(0.0, 2.0, .05, 1.0, "expr DAPI")
    types[f"T{i}"] = CellType(name=f"T{i}", Geometry=unnorm_geom(geom[k]), Expression=expr)

arch, kinds = ft._flexible_architecture(np.random.default_rng(1))
contexts = ["background"] + list(dict.fromkeys(kinds)); K = len(present)
comps = dict(contexts=contexts, pi=np.ones((len(contexts), K)) / K,
             allowed=np.ones((len(contexts), K), bool),
             struct_to_ctx=[contexts.index(kd) for kd in kinds], type_order=list(types))
# render the samples at the DENSITY the model inferred from the real crop
rtg = TissueGeometry()
rtg.MIN_DIST = narrow(rtg.MIN_DIST, rtg.MIN_DIST.lo, rtg.MIN_DIST.hi, pred_min_dist)
rtg.COVER    = narrow(rtg.COVER, rtg.COVER.lo, rtg.COVER.hi, pred_cover)
rtg.GROW     = narrow(rtg.GROW, rtg.GROW.lo, rtg.GROW.hi, pred_grow)
world = dict(PANEL=panel, TYPES=types, TG=rtg, DET=Detector(**cfg.DETECTOR),
             ARCH=arch, CELL=CellGeometry(), FRACTIONS=list(np.ones(K) / K), COMPOSITIONS=comps)
names = list(panel.Markers)
print(f"reconstructed panel {names}  |  {K} cell types")

In [ ]:
_mk = [m for m in names if m != "DAPI"]
r_ch = _mk[0] if len(_mk) > 0 else "DAPI"
g_ch = _mk[1] if len(_mk) > 1 else r_ch
ri = [names.index(r_ch), names.index(g_ch), names.index("DAPI")]   # two markers + DAPI(blue)
samples = [gd.render_frame(world, seed=s, pool_bank=cfg.POOL_BANK, with_artifacts=False)[0]
           for s in (1, 2, 3, 4)]
fig, ax = plt.subplots(1, 5, figsize=(18, 4))
ax[0].imshow(real_rgb); ax[0].set_title("REAL composite (reference)")
for j, s in enumerate(samples):
    rgb = np.stack([nrm(s[..., ri[0]]), nrm(s[..., ri[1]]), nrm(s[..., ri[2]])], -1)
    ax[j+1].imshow(rgb); ax[j+1].set_title(f"generated #{j+1}")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

### Notes
- Zero-shot transfer: trained only on synthetic data. Nuclei and cells are instance-segmented from
  Omnipose flows with `flow_decode.flow_to_instances` (short integration so a long cell's ridge stays
  one sink); a checkpoint trained with `--n-queries` adds the query-based per-cell head (section 4,
  last column; geojson export after section 4).
- Real crops are cut at the training CELL scale (`SCALE_MODE`, section 1): the raw window is sized
  so real nuclei span as many px as training nuclei; `"microns"` gives the nominal 0.325 um/px.
- The per-marker **strength** is a model INPUT (wired into the Stem and Trunk): since each image
  channel is normalised to [0,1] and thus brightness-invariant, strength is how the model tells a
  strongly-expressed channel from a dim / background-like one (low strength). It USES strength -- it
  does not ignore it.
- Generated samples re-sample placement and reconstruct a marker subset, so they show the inferred
  *biology*, not a pixel reconstruction. No ground truth exists on real data.
- The generated samples are now rendered at the model's **predicted** cell/structure density.
- Input domain: the staged **background-subtracted** pipeline output (empty regions are near-zero
  with sparse speckle); the generator now simulates MACSIMA backsub so synthetic and real match.
- Checkpoint: `runs/run12/model_backsub_final.pt` -- newest model, step 276000, trained in the background-subtracted domain (`step` printed on load).